# Master Person-Identity Pipeline

## Responsibility

Configure and run the complete two-stage workflow:

1. `01_collect_observations.ipynb` performs video tracking and face collection and stores the evidence.
2. `02_resolve_identities.ipynb` loads that evidence and resolves persistent logical person IDs.

Both child notebooks run in this kernel, so model-download messages, progress bars, exceptions, and final summaries remain visible here. The original `person_tracker.ipynb` is not used or modified.

## 1. Pipeline configuration

Edit this cell, then run the notebook from top to bottom. Frame values override their corresponding time values when they are not `None`.

In [ ]:
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY
NOTEBOOK_DIRECTORY = PROJECT_ROOT / "notebooks"

# Source and range
INPUT_VIDEO = PROJECT_ROOT / "input" / "source04_fixed.mp4"
START_SECONDS = 0.0
END_SECONDS = 5.0
START_FRAME = None
END_FRAME = None  # exclusive

# Give every independent extraction a unique name. Reusing a name overwrites it.
RUN_NAME = "source04-full"
SAVE_FACE_CROPS = True

# Detection, tracking, and face collection
FACE_DETECTION_HZ = 15
MAX_FACES_PER_FRAME = 0  # 0 means unlimited
YOLO_IMAGE_SIZE = 512
YOLO_MAX_DETECTIONS = 20
YOLO_CONFIDENCE = 0.25
YOLO_MODEL_NAME = "yolo11m.pt"
TRACKER_NAME = "deepocsort.yaml"
FACE_MODEL_NAME = "buffalo_l"
FACE_DETECTION_SIZE = (640, 640)
FACE_DETECTION_THRESHOLD = 0.35

# Logical identity resolution
MIN_FACE_QUALITY = 0.50
CLUSTER_EPS = 0.40
CLUSTER_MIN_SAMPLES = 4
MAX_REPRESENTATIVES_PER_TRACK = 12
ASSIGNMENT_SIMILARITY = 0.60
MERGE_SIMILARITY = 0.50
MERGE_TOP_K = 5
ANCHOR_SMOOTHING_WINDOW = 5
MIN_SEGMENT_ANCHORS = 2
MIN_SWITCH_JUMP_SCORE = 0.25
MAX_FALLBACK_SEARCH_SECONDS = 1.0
MIN_FALLBACK_OVERLAP = 0.15
TEMPORAL_DECAY_SECONDS = 1.0
FALLBACK_CONFIDENCE_SCALE = 0.75
MIN_FALLBACK_CONFIDENCE = 0.10
MIN_CONFLICT_CANDIDATE_SCORE = 0.10

## 2. Run collection and identity resolution

This can take a long time because the first child notebook performs model inference. If collection is interrupted, it saves the observations collected so far before raising the interruption.

### Stage 1 — Collect observations

In [ ]:
PERSON_TRACKER_MASTER = True

try:
    print("=" * 72)
    print("STAGE 1/2 — COLLECT OBSERVATIONS")
    print("=" * 72)
    get_ipython().run_line_magic(
        "run",
        f'-i "{NOTEBOOK_DIRECTORY / "01_collect_observations.ipynb"}"',
    )
finally:
    PERSON_TRACKER_MASTER = False

### Stage 2 — Resolve logical identities

In [ ]:
PERSON_TRACKER_MASTER = True

try:
    print("=" * 72)
    print("STAGE 2/2 — RESOLVE LOGICAL IDENTITIES")
    print("=" * 72)
    get_ipython().run_line_magic(
        "run",
        f'-i "{NOTEBOOK_DIRECTORY / "02_resolve_identities.ipynb"}"',
    )
finally:
    PERSON_TRACKER_MASTER = False

## 3. Result locations

In [ ]:
print(f"Run directory: {RUN_DIRECTORY}")
print(f"Observation manifest: {RUN_DIRECTORY / 'manifest.json'}")
print(f"Tracking records: {RUN_DIRECTORY / 'tracks.jsonl'}")
print(f"Face embeddings: {RUN_DIRECTORY / 'face_embeddings.npy'}")
print(f"Identity assignments: {RUN_DIRECTORY / 'identities.jsonl'}")
print(f"Identity events: {RUN_DIRECTORY / 'identity_events.json'}")
print(f"Identity summary: {RUN_DIRECTORY / 'identity_summary.json'}")